# Entropy lens

How does vocabulary uncertainty evolve through recurrent depth? Compare the shared
paper models with their matched feedforward controls, using the
[Entropy-Lens](https://arxiv.org/abs/2502.16570) readout: final normalization and the
vocabulary head applied to each block's output, followed by Shannon entropy.
For Parcae, prelude and core outputs first pass through the learned projection C;
asterisks identify these panels in the ordinary-lens figures.

Ordinary and vocabulary-normalized entropy each have main and appendix figures.
For models with a coda, an additional **entropy-coda-lens** reads each prelude/core
state through the trained coda before normalization and vocabulary decoding.
Both coda-lens figures use all eight model panels, with “No coda” for Ouro.

Quick mode uses one seeded example and saves only to the run directory; full mode
uses 256 and also refreshes paper figures. An explicit `limit` changes sample
count, never export policy. Both modes use the same models and rollout lengths;
SLURM selects full mode.

In [ ]:
mode = "quick"
dataset = "gsm8k"  # "gsm8k" or "hellaswag"
# Shared main/appendix groups, including Parcae 770M and its matched control.
recurrences = 32
ouro_recurrences = 8
limit = None  # 1 (quick), 256 (full); 0 selects the entire split.
seed = 42
device = "auto"
dtype = "auto"
token_chunk_size = 32  # Bound the temporary token-by-vocabulary readout.
output_dir = None

cache_dir = None  # Defaults to LOOPED_INTERNALS_CACHE_DIR or /data-gauss/hugbla/looped-internals-cache.
cache_mode = None  # Environment default; "reuse", "require" (no inference), or "off".
analysis_version = 1  # Bump when the entropy reduction changes.
source_run = None  # Optional completed run directory with matching scientific settings.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from dataclasses import asdict
from functools import partial
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import torch

from experiments.cache import ExperimentCache
from experiments.paper_models import PAPER_MODEL_GROUPS, paper_model_pairs
from experiments.readout_support import collect_residual_readouts
from experiments.saved_runs import SavedRun
from experiments.support import (
    BASELINE_FOR,
    COLORMAPS,
    MODEL_LABELS,
    add_depth_colorbar,
    add_prelude_coda_connectors,
    add_prelude_coda_legend,
    configure_experiment,
    export_paper_figures,
    model_subplots,
    save_figure,
    save_results,
)
from looped_internals.evaluation import coda_hidden_states
from looped_internals.evaluation.logit_lens import logit_lens

plt.style.use(ROOT / "plots" / "matlab.mplstyle")

# Resolve shared model groups after parameter overrides.
recurrent_pairs, figure_pairs = paper_model_pairs(
    recurrences,
    [
        (name, ouro_recurrences if name == "ouro_1_4b" else recurrences)
        for name in dict.fromkeys(PAPER_MODEL_GROUPS["all"])
    ],
)
baseline_pairs = list(
    dict.fromkeys((BASELINE_FOR[name], 1) for name, _ in recurrent_pairs)
)
model_recurrences = recurrent_pairs + baseline_pairs
projected_pairs = [pair for pair in model_recurrences if pair[0].startswith("parcae_")]
# Parcae controls also have a trained projection and coda; Ouro has no coda.
coda_pairs = [pair for pair in recurrent_pairs if pair[0] != "ouro_1_4b"] + [
    pair for pair in baseline_pairs if pair in projected_pairs
]
if type(token_chunk_size) is not int or token_chunk_size < 1:
    raise ValueError("token_chunk_size must be a positive integer")

sample_limit = limit if limit is not None else (1 if mode == "quick" else 256)
configs = {
    (name, recurrences): configure_experiment(
        experiment="entropy_lens",
        dataset=dataset,
        mode=mode,
        models=[name],
        num_steps=recurrences,
        limit=sample_limit,
        seed=seed,
        device=device,
        dtype=dtype,
        output_dir=output_dir,
        compute_residual_entropy=False,
    )
    for name, recurrences in model_recurrences
}
config = next(iter(configs.values()))

result_cache = ExperimentCache(cache_dir, cache_mode)
# Only Parcae requests change: all other models retain their existing cache keys.
readout_options = {
    pair: {
        "compute_coda": pair in coda_pairs,
        "token_chunk_size": token_chunk_size,
        **(
            {"parcae_projection": "prelude_and_core"} if pair in projected_pairs else {}
        ),
    }
    for pair in model_recurrences
}

In [ ]:
def mean_vocabulary_entropy(bridge, residual):
    """Compute entropy per position, then average over this example's tokens."""
    total = residual.new_zeros((), dtype=torch.float32)
    for hidden in residual.split(token_chunk_size, dim=1):
        logits = logit_lens(bridge, hidden)  # Final norm + head, float32 logits.
        log_p = logits.log_softmax(dim=-1)
        entropy = -(log_p.exp() * log_p).sum(dim=-1)
        total += entropy.sum()
    return {
        "entropy_nats": (total / residual.shape[1]).item(),
        "vocab_size": logits.shape[-1],
    }


def summarize_readouts(
    bridge,
    residual,
    layer,
    structure,
    *,
    input_ids,
    compute_coda,
    project_pre_coda=False,
):
    core_layers = [
        v.layer_index for v in structure.virtual_layers() if v.iteration is not None
    ]
    last_layer = sum(group.n_layers for group in structure.groups) - 1
    has_coda = bool(core_layers) and max(core_layers) < last_layer
    before_coda = has_coda and layer.layer_index <= max(core_layers)
    # This also defines a hypothetical projected readout of the prelude states.
    hidden = (
        bridge.core_projection(residual)
        if project_pre_coda and before_coda
        else residual
    )
    values = mean_vocabulary_entropy(bridge, hidden)
    values["coda_entropy_nats"] = None
    # Omit observed coda states, which have already passed through C and the coda.
    if compute_coda and before_coda:
        coda_output = coda_hidden_states(
            bridge,
            hidden,
            input_ids=input_ids,
            # Projected Parcae states must enter the suffix without a second C.
            from_core=layer.iteration is not None and not project_pre_coda,
        )
        values["coda_entropy_nats"] = mean_vocabulary_entropy(bridge, coda_output)[
            "entropy_nats"
        ]
    return values


def aggregate_entropy(examples, info, field="entropy_nats"):
    """Weight examples equally, retaining their layer and recurrence identities."""
    selected = [i for i, value in enumerate(examples[0]) if value[field] is not None]
    values = np.asarray([[example[i][field] for i in selected] for example in examples])
    vocab_size = examples[0][0]["vocab_size"]
    layout = [info["virtual_layers"][i] for i in selected]
    depth_max = max(layer["layer_index"] for layer in info["virtual_layers"])
    return {
        "layers": np.asarray([layer["layer_index"] for layer in layout]),
        "recurrences": np.asarray([layer["iteration"] for layer in layout]),
        "recurrent_layers": sorted(
            {layer["layer_index"] for layer in layout if layer["iteration"] is not None}
        ),
        "vocab_size": vocab_size,
        "depth_max": depth_max,
        "mean": values.mean(axis=0),
        "std": values.std(axis=0, ddof=1 if len(examples) > 1 else 0),
    }

## Generate and reduce

The shared groups supply Ouro at eight recurrences and the other recurrent models
at 32, plus each matched control at one pass. Each pair runs once on the same
seeded examples from the packaged likelihood task. Parcae 770M is paired with
its trained one-recurrence control; 140M and 370M keep their own matched controls.

Inputs include context and reference continuation with native checkpoint tokenization,
matching the stages-of-inference experiment. We measure next-token distributions
at **all input positions**, including any native BOS and the final position. This is
a teacher-forced profile, rather than the paper's average over sampled generations.
No tokens are generated here.

Only block-output hooks are installed. Each output is immediately decoded in token
chunks and reduced to mean entropies; no residual history, attention matrices,
spectral entropy, or pairwise comparisons are collected. Each model is released
after its examples. Recurrent models with a coda and the Parcae controls compute the extra
readout; the other controls supply ordinary entropy only. The coda lens reruns the coda on
each prelude/core output within that same inference pass. Coda attention sees the
**complete sequence**; only the subsequent vocabulary projection is chunked. Auxiliary coda applications
are excluded from the observed forward-pass layer sequence. `overview.pt` retains
per-example curves and input/model metadata; all later cells reuse these summaries
without inference.

In [ ]:
settings = asdict(config)
for key in ("models", "num_steps", "compute_residual_entropy", "include_baselines"):
    del settings[key]
settings.update(
    model_recurrences=model_recurrences,
    figure_pairs=figure_pairs,
    coda_pairs=coda_pairs,
    output_dir=str(config.output_dir),
    token_chunk_size=token_chunk_size,
    readout="final_norm_then_unembedding",
    parcae_projection="prelude_and_core",
    coda_readout="coda_suffix_then_final_norm_and_unembedding",
    entropy_units="nats",
    token_selection="all_input_positions_including_native_special_tokens",
    input_protocol="teacher_forced_context_and_reference_continuation",
    versions={
        name: version(name) for name in ("torch", "transformer-lens", "transformers")
    },
)
settings["cache"] = result_cache.settings
previous = SavedRun(source_run, settings, analysis_version=analysis_version)
previous.check_pairs(settings)
settings.update(analysis_version=analysis_version, source_run=source_run)
run_dir = save_results({}, settings, config.output_dir)
summaries, run_info = {}, {}
for pair, pair_config in configs.items():
    if pair in previous:
        summaries[pair] = previous.data["summaries"][pair]
        run_info[pair] = previous.info(pair)
        continue
    summaries[pair], run_info[pair] = collect_residual_readouts(
        pair_config,
        partial(
            summarize_readouts,
            compute_coda=pair in coda_pairs,
            project_pre_coda=pair in projected_pairs,
        ),
        cache=result_cache,
        analysis_version=analysis_version,
        readout_options=readout_options[pair],
    )

profiles = {
    pair: aggregate_entropy(examples, run_info[pair])
    for pair, examples in summaries.items()
}
coda_profiles = {
    pair: aggregate_entropy(examples, run_info[pair], "coda_entropy_nats")
    for pair, examples in summaries.items()
    if any(value["coda_entropy_nats"] is not None for value in examples[0])
}
torch.save(
    {"summaries": summaries, "run": run_info, "figure_pairs": figure_pairs},
    run_dir / "overview.pt",
)
result_cache.write_manifest(run_dir)

## What the curves measure

For block output $h_{s,t}$ at virtual layer $s$ and input position $t$,
$p_{s,t}=\mathrm{softmax}(U\,\mathrm{Norm}_{\mathrm{final}}(h_{s,t}))$ and
$H_{s,t}=-\sum_v p_{s,t,v}\log p_{s,t,v}$, with temperature one.
For Parcae and its controls, replace $h_{s,t}$ by $C h_{s,t}$ at every prelude
and core point, where $C$ is the checkpoint's learned core-output projection.
Observed coda states are read directly. Prelude projection is a hypothetical
readout: native execution first runs input injection and the recurrent core.
An **asterisk on the ordinary-lens panel title** marks this Parcae convention.
All other models use raw block outputs. Final normalization is applied once;
there is no embedding-only point. The final coda endpoint still reproduces the
model's vocabulary readout, and ordinary intermediate points never run the coda.

| Quantity | Meaning |
| --- | --- |
| Entropy (nats) | Shannon entropy of each vocabulary distribution; average over positions within an example, then equally over examples. This is the mean of entropies, not entropy of a pooled distribution. |
| Normalized entropy | $H/\log V$, where $V$ is the number of output-head entries, including any padded entries. Zero is a point mass and one is uniform. |
| Shading | Mean $\pm$ one sample standard deviation across example means; zero for a single example. Bands are not confidence intervals. |
| Layer depth (%) | Recorded physical layer index divided by its maximum. Prelude, core and coda occupy their own positions; each recurrence overlays the same core span. Each control uses its own 0–100% depth axis. |

Colors follow the established **summer** recurrence scale (early green to late yellow).
Prelude is orange, coda black, and the matched feedforward control dashed red.
Stage boundaries come from recorded layer metadata. Faint dashed grey links join
the prelude to the first recurrence and the final recurrence to the coda.

Both main and appendix groups have a figure in nats and a normalized version.
Normalization removes the vocabulary-size upper bound using the same data. This
scaling neither aligns tokenizers nor makes entropy an accuracy measure. Low entropy indicates a concentrated decoded distribution,
which can still be wrong; the profiles alone do not establish convergence.

The **coda lens** runs the trained coda before final normalization and vocabulary
readout. Huginn first applies its native pre-coda normalization; the retrofits
send the raw state directly into their coda. For Parcae, both prelude and core
states first pass through $C$, exactly once, then through the coda. This projection
is treated as part of the coda readout, so these titles have no asterisk.
Parcae's coda also receives the original input IDs for its trained value embeddings. All use the original causal positions and a fresh, uncached coda
evaluation. Thus the last recurrent block's coda-lens prediction reproduces the
ordinary final prediction. Applying
this suffix earlier (including to prelude outputs) is an intermediate readout,
not the model's actual execution at that depth.

Coda-lens figures omit observed coda states and retain the original physical-depth
scale, leaving their positions empty. Parcae's dashed red controls use the same
projected coda lens, also omitting their observed coda states; the other controls
retain their ordinary entropy profiles. Both figures show all eight models in
paper order.
Ouro's panel is marked “No coda”.

In [ ]:
def recurrence_groups(data):
    layers, core = data["layers"], data["recurrent_layers"]
    if not core:
        return {"Feedforward": np.arange(len(layers))}
    groups = {}
    if np.any(layers < min(core)):
        groups["Prelude"] = np.flatnonzero(layers < min(core))
    recurrences = data["recurrences"]
    for recurrence in sorted({r for r in recurrences if r is not None}):
        groups[f"R{recurrence}"] = np.flatnonzero(recurrences == recurrence)
    if np.any(layers > max(core)):
        groups["Coda"] = np.flatnonzero(layers > max(core))
    return groups


def draw_entropy(ax, data, indices, *, normalized=False, **style):
    x = data["layers"][indices] / max(data["depth_max"], 1) * 100
    scale = np.log(data["vocab_size"]) if normalized else 1
    mean, std = data["mean"][indices] / scale, data["std"][indices] / scale
    (line,) = ax.plot(x, mean, marker="." if len(indices) == 1 else None, **style)
    ax.fill_between(x, mean - std, mean + std, color=line.get_color(), alpha=0.12)
    return line


def plot_entropy_profiles(group, filename, *, normalized=False, coda=False):
    selected_profiles = coda_profiles if coda else profiles
    pairs = figure_pairs[group]
    fig, axes = model_subplots(len(pairs), height_per_subplot=1.9)
    cmap = plt.colormaps[COLORMAPS["recurrences"]]
    for index, (ax, pair) in enumerate(zip(axes, pairs, strict=True)):
        if index:
            ax.sharey(axes[0])
        ax.set(
            title=MODEL_LABELS[pair[0]]
            + ("*" if not coda and pair in projected_pairs else ""),
            xlabel="Layer depth (%)",
            xlim=(-3, 103),
            xticks=[0, 50, 100],
        )
        if index % 4 == 0:
            ax.set_ylabel("Entropy / log V" if normalized else "Entropy (nats)")
        else:
            ax.tick_params(labelleft=False)
        if coda and pair not in coda_pairs:
            ax.text(
                0.5, 0.5, "No coda", ha="center", va="center", transform=ax.transAxes
            )
            continue
        data = selected_profiles[pair]
        groups = recurrence_groups(data)
        n_recurrences = sum(key.startswith("R") for key in groups)
        lines = {}
        for key, indices in groups.items():
            if key in {"Prelude", "Coda"}:
                color, label = ("orange" if key == "Prelude" else "black"), key
            else:
                color, label = cmap(int(key[1:]) / max(n_recurrences - 1, 1)), None
            lines[key] = draw_entropy(
                ax,
                data,
                indices,
                normalized=normalized,
                color=color,
                label=label,
                linewidth=1.5,
            )
        baseline_pair = (BASELINE_FOR[pair[0]], 1)
        baseline = (
            coda_profiles[baseline_pair]
            if coda and baseline_pair in coda_pairs
            else profiles[baseline_pair]
        )
        draw_entropy(
            ax,
            baseline,
            np.arange(len(baseline["layers"])),
            normalized=normalized,
            color="red",
            linestyle="--",
            linewidth=1.5,
            label="Control",
        )
        add_prelude_coda_connectors(ax, lines)
    if normalized:
        axes[0].set_ylim(0, 1.02)
    else:
        axes[0].set_ylim(bottom=0)
    add_prelude_coda_legend(fig, include_coda=not coda)
    add_depth_colorbar(fig, axes, "recurrence")
    save_figure(fig, run_dir, filename, bbox_inches=None)

## Entropy through depth

Each full-width figure contains four models and their matched controls.


In [ ]:
plot_entropy_profiles("main", "entropy_lens_main")

In [ ]:
plot_entropy_profiles("appendix", "entropy_lens_appendix")

## Vocabulary-normalized entropy

Divide every model and control profile by its own maximum entropy $\log V$.


In [ ]:
plot_entropy_profiles("main", "entropy_lens_normalized_main", normalized=True)

In [ ]:
plot_entropy_profiles("appendix", "entropy_lens_normalized_appendix", normalized=True)

## Entropy through the trained coda

The main models form the first row and appendix models the second. Ouro has no
coda. Other panels read each prelude/core state through the trained suffix,
retaining the original physical-depth coordinates and omitting observed coda states.
Parcae panels apply the same $C$-then-coda readout to recurrent models and controls.


In [ ]:
plot_entropy_profiles("all", "entropy_coda_lens_all", coda=True)

In [ ]:
plot_entropy_profiles(
    "all", "entropy_coda_lens_normalized_all", coda=True, normalized=True
)

In [ ]:
print(run_dir)
paper_dir = export_paper_figures(config, run_dir, dataset=config.dataset)
if paper_dir is not None:
    print(f"Saved paper figures to {paper_dir}")